# PDF Editor

In [ ]:
# ============================================================
# Professional PDF Tools GUI in Jupyter Notebook / VS Code
# Final Improved Version - Full Code in One Cell
#
# Functions:
# 1) Merge PDFs
# 2) Split PDF
# 3) Compress PDFs
# 4) JPG to PDF
# 5) PDF to JPG
#
# New Feature:
# - Reorder selected files using ↑ and ↓ buttons
#
# Workflow:
# 1. Click a tool button.
# 2. Select file(s) from your PC.
# 3. The selected files will appear below.
# 4. Reorder files using ↑ and ↓ if needed.
# 5. Click Process to run the selected action.
# 6. Click Download File and choose where to save the result.
# ============================================================

import os
import sys
import shutil
import zipfile
import tempfile
from pathlib import Path
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display

import fitz  # PyMuPDF
from PIL import Image

import tkinter as tk
from tkinter import filedialog


# ============================================================
# Global State
# ============================================================

current_action = None
selected_files = []
processed_output_path = None

TEMP_ROOT = Path(tempfile.gettempdir()) / "pdf_tools_gui_outputs"
TEMP_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# Helper Functions
# ============================================================

def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def create_hidden_tk_root():
    """
    Create hidden tkinter window for normal Windows browsing dialogs.
    """
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    return root


def file_size_mb(path):
    path = Path(path)

    if path.exists():
        return path.stat().st_size / (1024 * 1024)

    return 0


def set_message(message, kind="info"):
    """
    Display clean status messages inside the GUI.
    No technical file paths are displayed.
    """
    colors = {
        "info": {
            "bg": "#eff6ff",
            "border": "#93c5fd",
            "text": "#1e3a8a",
            "icon": "ℹ️"
        },
        "success": {
            "bg": "#ecfdf5",
            "border": "#86efac",
            "text": "#166534",
            "icon": "✅"
        },
        "warning": {
            "bg": "#fffbeb",
            "border": "#fcd34d",
            "text": "#92400e",
            "icon": "⚠️"
        },
        "error": {
            "bg": "#fef2f2",
            "border": "#fca5a5",
            "text": "#991b1b",
            "icon": "❌"
        }
    }

    style = colors.get(kind, colors["info"])

    message_html.value = f"""
    <div style="
        background: {style['bg']};
        border: 1px solid {style['border']};
        color: {style['text']};
        padding: 11px 13px;
        border-radius: 10px;
        font-family: Arial;
        width: 650px;
        margin: 10px 0;
        font-size: 14px;
    ">
        {style['icon']} {message}
    </div>
    """


def reset_processed_output():
    """
    Remove the previous Download File button when the user changes action/files/order.
    """
    global processed_output_path

    processed_output_path = None
    download_box.children = []


def select_pdf_files(multiple=True, title="Select PDF file(s)"):
    """
    Open browsing window to select PDF file(s).
    """
    root = create_hidden_tk_root()

    try:
        if multiple:
            files = filedialog.askopenfilenames(
                title=title,
                filetypes=[("PDF files", "*.pdf")]
            )
            return list(files)

        file = filedialog.askopenfilename(
            title=title,
            filetypes=[("PDF files", "*.pdf")]
        )

        return [file] if file else []

    finally:
        root.destroy()


def select_jpg_files(title="Select JPG file(s)"):
    """
    Open browsing window to select JPG/JPEG image file(s).
    """
    root = create_hidden_tk_root()

    try:
        files = filedialog.askopenfilenames(
            title=title,
            filetypes=[
                ("JPG files", "*.jpg *.jpeg"),
                ("All files", "*.*")
            ]
        )

        return list(files)

    finally:
        root.destroy()


def choose_save_file(default_name, extension, title="Save file as"):
    """
    Open Save As window to choose where to save the output.
    """
    root = create_hidden_tk_root()

    try:
        save_path = filedialog.asksaveasfilename(
            title=title,
            defaultextension=extension,
            initialfile=default_name,
            filetypes=[
                (f"{extension.upper()} files", f"*{extension}"),
                ("All files", "*.*")
            ]
        )

        return save_path

    finally:
        root.destroy()


def zip_folder(folder_path):
    """
    Convert a folder into a ZIP file.
    Used for Split PDF and PDF to JPG.
    """
    folder_path = Path(folder_path)
    zip_path = TEMP_ROOT / f"{folder_path.name}.zip"

    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file in folder_path.rglob("*"):
            if file.is_file():
                zipf.write(file, file.relative_to(folder_path.parent))

    return zip_path


# ============================================================
# GUI Update Functions
# ============================================================

def update_status_text():
    if current_action is None:
        status_html.value = """
        <div style="
            padding: 12px;
            background: #eef2ff;
            border-radius: 10px;
            font-family: Arial;
            margin-top: 10px;
            width: 650px;
        ">
            Choose a tool above to start.
        </div>
        """
    else:
        status_html.value = f"""
        <div style="
            padding: 12px;
            background: #eef2ff;
            border-radius: 10px;
            font-family: Arial;
            margin-top: 10px;
            width: 650px;
        ">
            <b>Current action:</b> {current_action}<br>
            <b>Selected files:</b> {len(selected_files)}
        </div>
        """


def update_file_list_display():
    file_rows = []

    if not selected_files:
        file_list_box.children = [
            widgets.HTML(
                value="""
                <div style="
                    padding: 12px;
                    background: #fff7ed;
                    border: 1px solid #fed7aa;
                    border-radius: 10px;
                    font-family: Arial;
                    width: 420px;
                ">
                    No files selected yet.
                </div>
                """
            )
        ]

        action_buttons_box.children = []
        update_status_text()
        return

    for index, file_path in enumerate(selected_files):
        path_obj = Path(file_path)
        size = file_size_mb(path_obj)

        file_label = widgets.HTML(
            value=f"""
            <div style="
                padding: 10px 12px;
                background: #f9fafb;
                border: 1px solid #e5e7eb;
                border-radius: 10px;
                font-family: Arial;
                min-width: 360px;
                max-width: 520px;
                font-size: 14px;
            ">
                📄 <b>{index + 1}. {path_obj.name}</b>
                <span style="color: #6b7280;">({size:.2f} MB)</span>
            </div>
            """
        )

        move_up_button = widgets.Button(
            description="↑",
            tooltip="Move this file up",
            button_style="",
            disabled=(index == 0),
            layout=widgets.Layout(
                width="30px",
                height="30px",
                padding="0px",
                margin="0px"
            )
        )

        move_down_button = widgets.Button(
            description="↓",
            tooltip="Move this file down",
            button_style="",
            disabled=(index == len(selected_files) - 1),
            layout=widgets.Layout(
                width="30px",
                height="30px",
                padding="0px",
                margin="0px"
            )
        )

        remove_button = widgets.Button(
            description="×",
            tooltip="Remove this file",
            button_style="danger",
            layout=widgets.Layout(
                width="30px",
                height="30px",
                padding="0px",
                margin="0px"
            )
        )

        def move_up_clicked(b, file_index=index):
            move_file_up(file_index)

        def move_down_clicked(b, file_index=index):
            move_file_down(file_index)

        def remove_clicked(b, file_to_remove=file_path):
            remove_selected_file(file_to_remove)

        move_up_button.on_click(move_up_clicked)
        move_down_button.on_click(move_down_clicked)
        remove_button.on_click(remove_clicked)

        row = widgets.HBox(
            [file_label, move_up_button, move_down_button, remove_button],
            layout=widgets.Layout(
                gap="6px",
                align_items="center",
                margin="5px 0"
            )
        )

        file_rows.append(row)

    file_list_box.children = file_rows

    upload_more_button = widgets.Button(
        description="Upload More Files",
        icon="upload",
        button_style="info",
        layout=widgets.Layout(width="220px", height="42px")
    )

    process_button = widgets.Button(
        description="Process",
        icon="play",
        button_style="success",
        layout=widgets.Layout(width="160px", height="42px")
    )

    clear_files_button = widgets.Button(
        description="Clear Files",
        icon="trash",
        button_style="warning",
        layout=widgets.Layout(width="160px", height="42px")
    )

    upload_more_button.on_click(lambda b: upload_more_files())
    process_button.on_click(lambda b: process_current_action())
    clear_files_button.on_click(lambda b: clear_selected_files())

    action_buttons_box.children = [
        widgets.HBox(
            [upload_more_button, process_button, clear_files_button],
            layout=widgets.Layout(gap="12px", margin="12px 0")
        )
    ]

    update_status_text()


def show_download_button():
    """
    Show only the Download File button.
    This is the last visible element in the GUI.
    """
    download_button = widgets.Button(
        description="Download File",
        icon="download",
        button_style="primary",
        layout=widgets.Layout(width="220px", height="45px")
    )

    download_button.on_click(lambda b: download_processed_file())

    download_box.children = [
        widgets.HBox(
            [download_button],
            layout=widgets.Layout(gap="12px", margin="12px 0 0 0")
        )
    ]


# ============================================================
# File List Control Functions
# ============================================================

def move_file_up(index):
    global selected_files

    if index <= 0:
        return

    selected_files[index - 1], selected_files[index] = selected_files[index], selected_files[index - 1]

    reset_processed_output()
    update_file_list_display()
    set_message("File moved up.", "info")


def move_file_down(index):
    global selected_files

    if index >= len(selected_files) - 1:
        return

    selected_files[index + 1], selected_files[index] = selected_files[index], selected_files[index + 1]

    reset_processed_output()
    update_file_list_display()
    set_message("File moved down.", "info")


def remove_selected_file(file_path):
    global selected_files

    selected_files = [f for f in selected_files if f != file_path]

    reset_processed_output()
    update_file_list_display()

    set_message("File removed.", "warning")


def clear_selected_files():
    global selected_files

    selected_files = []

    reset_processed_output()
    update_file_list_display()

    set_message("Selected files cleared.", "warning")


def add_files_to_selection(new_files):
    global selected_files

    reset_processed_output()

    if not new_files:
        set_message("No files selected.", "warning")
        return

    added_count = 0

    for file in new_files:
        if file and file not in selected_files:
            selected_files.append(file)
            added_count += 1

    update_file_list_display()

    if added_count > 0:
        set_message(f"{added_count} file(s) added.", "success")
    else:
        set_message("Selected file(s) already exist in the list.", "warning")


# ============================================================
# Action Selection
# ============================================================

def start_action(action_name):
    global current_action, selected_files

    current_action = action_name
    selected_files = []

    reset_processed_output()
    update_status_text()

    if action_name == "Merge PDFs":
        files = select_pdf_files(
            multiple=True,
            title="Select PDF files to merge"
        )

    elif action_name == "Split PDF":
        files = select_pdf_files(
            multiple=False,
            title="Select one PDF file to split"
        )

    elif action_name == "Compress PDFs":
        files = select_pdf_files(
            multiple=True,
            title="Select PDF file(s) to compress"
        )

    elif action_name == "JPG to PDF":
        files = select_jpg_files(
            title="Select JPG files to convert to PDF"
        )

    elif action_name == "PDF to JPG":
        files = select_pdf_files(
            multiple=False,
            title="Select one PDF file to convert to JPG"
        )

    else:
        files = []

    add_files_to_selection(files)


def upload_more_files():
    if current_action is None:
        set_message("Choose an action first.", "warning")
        return

    if current_action == "Merge PDFs":
        files = select_pdf_files(
            multiple=True,
            title="Select more PDF files to merge"
        )

    elif current_action == "Split PDF":
        files = select_pdf_files(
            multiple=False,
            title="Select PDF file"
        )

    elif current_action == "Compress PDFs":
        files = select_pdf_files(
            multiple=True,
            title="Select more PDF files to compress"
        )

    elif current_action == "JPG to PDF":
        files = select_jpg_files(
            title="Select more JPG files"
        )

    elif current_action == "PDF to JPG":
        files = select_pdf_files(
            multiple=False,
            title="Select PDF file"
        )

    else:
        files = []

    if current_action in ["Split PDF", "PDF to JPG"]:
        if files:
            selected_files.clear()
            add_files_to_selection(files)
            set_message("Only one file is allowed for this action.", "warning")
        return

    add_files_to_selection(files)


# ============================================================
# Processing Controller
# ============================================================

def process_current_action():
    if current_action is None:
        set_message("Choose an action first.", "warning")
        return

    if not selected_files:
        set_message("No files selected.", "warning")
        return

    reset_processed_output()

    try:
        if current_action == "Merge PDFs":
            process_merge_pdfs()

        elif current_action == "Split PDF":
            process_split_pdf()

        elif current_action == "Compress PDFs":
            process_compress_pdfs()

        elif current_action == "JPG to PDF":
            process_jpg_to_pdf()

        elif current_action == "PDF to JPG":
            process_pdf_to_jpg()

        else:
            set_message("Unknown action.", "error")

    except Exception:
        set_message("An error happened during processing. Please check your files.", "error")


# ============================================================
# Processing Functions
# ============================================================

def process_merge_pdfs():
    """
    Merge PDFs using PyMuPDF.
    This method is strong for real-world PDF files.
    The merge order follows the visible order in the GUI.
    """
    global processed_output_path

    if len(selected_files) < 2:
        set_message("Please select at least 2 PDF files for merging.", "warning")
        return

    output_file = TEMP_ROOT / f"merged_{timestamp()}.pdf"

    merged_doc = fitz.open()
    failed_files = []

    for pdf_file in selected_files:
        input_path = Path(pdf_file)

        try:
            src_doc = fitz.open(str(input_path))

            if len(src_doc) == 0:
                failed_files.append(input_path.name)
                src_doc.close()
                continue

            merged_doc.insert_pdf(src_doc)
            src_doc.close()

        except Exception:
            failed_files.append(input_path.name)

    if len(merged_doc) == 0:
        merged_doc.close()
        set_message("No pages were merged. Please check your PDF files.", "error")
        return

    merged_doc.save(
        str(output_file),
        garbage=4,
        deflate=True,
        clean=True
    )

    merged_doc.close()

    processed_output_path = output_file

    if failed_files:
        set_message("PDF files merged, but some files could not be added.", "warning")
    else:
        set_message("PDF files merged successfully.", "success")

    show_download_button()


def process_split_pdf():
    """
    Split one PDF into separate page PDFs.
    Output is prepared as a ZIP file.
    """
    global processed_output_path

    if len(selected_files) != 1:
        set_message("Please select only one PDF file for splitting.", "warning")
        return

    input_pdf = Path(selected_files[0])
    output_folder = TEMP_ROOT / f"{input_pdf.stem}_split_{timestamp()}"
    output_folder.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(str(input_pdf))
    total_pages = len(doc)

    for page_index in range(total_pages):
        new_doc = fitz.open()
        new_doc.insert_pdf(doc, from_page=page_index, to_page=page_index)

        page_file = output_folder / f"{input_pdf.stem}_page_{page_index + 1:03d}.pdf"

        new_doc.save(
            str(page_file),
            garbage=4,
            deflate=True,
            clean=True
        )

        new_doc.close()

    doc.close()

    processed_output_path = zip_folder(output_folder)

    set_message(f"PDF split successfully into {total_pages} page(s).", "success")
    show_download_button()


def process_compress_pdfs():
    """
    Compress one or more PDF files.
    If multiple files are selected, output is prepared as ZIP.
    """
    global processed_output_path

    output_folder = TEMP_ROOT / f"compressed_pdfs_{timestamp()}"
    output_folder.mkdir(parents=True, exist_ok=True)

    successful_files = 0

    for pdf_file in selected_files:
        input_path = Path(pdf_file)
        output_path = output_folder / f"{input_path.stem}_compressed.pdf"

        try:
            doc = fitz.open(str(input_path))

            doc.save(
                str(output_path),
                garbage=4,
                deflate=True,
                clean=True
            )

            doc.close()
            successful_files += 1

        except Exception:
            pass

    if successful_files == 0:
        set_message("No PDF files were compressed.", "error")
        return

    pdf_outputs = list(output_folder.glob("*.pdf"))

    if successful_files == 1:
        processed_output_path = pdf_outputs[0]
    else:
        processed_output_path = zip_folder(output_folder)

    set_message("Compression completed. Size reduction depends on the original PDF.", "success")
    show_download_button()


def process_jpg_to_pdf():
    """
    Convert selected JPG/JPEG images into one PDF file.
    The PDF page order follows the visible order in the GUI.
    """
    global processed_output_path

    output_file = TEMP_ROOT / f"jpg_to_pdf_{timestamp()}.pdf"
    images = []

    for image_file in selected_files:
        try:
            img = Image.open(image_file)

            if img.mode != "RGB":
                img = img.convert("RGB")

            images.append(img)

        except Exception:
            pass

    if not images:
        set_message("No valid JPG images found.", "error")
        return

    first_image = images[0]
    other_images = images[1:]

    first_image.save(
        output_file,
        save_all=True,
        append_images=other_images
    )

    for img in images:
        img.close()

    processed_output_path = output_file

    set_message("JPG files converted to PDF successfully.", "success")
    show_download_button()


def process_pdf_to_jpg():
    """
    Convert each page of a PDF into JPG images.
    Output is prepared as a ZIP file.
    """
    global processed_output_path

    if len(selected_files) != 1:
        set_message("Please select only one PDF file for PDF to JPG.", "warning")
        return

    input_pdf = Path(selected_files[0])
    output_folder = TEMP_ROOT / f"{input_pdf.stem}_jpg_{timestamp()}"
    output_folder.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(str(input_pdf))
    total_pages = len(doc)

    zoom = 2.0
    matrix = fitz.Matrix(zoom, zoom)

    for page_index in range(total_pages):
        page = doc.load_page(page_index)
        pix = page.get_pixmap(matrix=matrix, alpha=False)

        image_file = output_folder / f"{input_pdf.stem}_page_{page_index + 1:03d}.jpg"
        pix.save(str(image_file))

    doc.close()

    processed_output_path = zip_folder(output_folder)

    set_message(f"PDF converted to {total_pages} JPG image(s).", "success")
    show_download_button()


# ============================================================
# Download Function
# ============================================================

def download_processed_file():
    """
    Save the processed output to the user's chosen location.
    """
    if processed_output_path is None:
        set_message("No processed file available yet.", "warning")
        return

    source = Path(processed_output_path)

    if not source.exists():
        set_message("Processed output was not found.", "error")
        return

    extension = source.suffix
    default_name = source.name

    save_path = choose_save_file(
        default_name=default_name,
        extension=extension,
        title="Choose where to save the output file"
    )

    if not save_path:
        set_message("Save cancelled.", "warning")
        return

    try:
        shutil.copy2(source, save_path)
        set_message("File saved successfully.", "success")

    except Exception:
        set_message("Error while saving the file.", "error")


# ============================================================
# GUI Layout
# ============================================================

title = widgets.HTML(
    value="""
    <div style="
        background: linear-gradient(90deg, #111827, #374151);
        color: white;
        padding: 18px;
        border-radius: 12px;
        text-align: center;
        font-family: Arial;
        margin-bottom: 15px;
        width: 650px;
    ">
        <h1 style="margin: 0;">📄 Professional PDF Tools</h1>
        <p style="margin: 6px 0 0 0; font-size: 15px;">
            Merge • Split • Compress • JPG to PDF • PDF to JPG
        </p>
    </div>
    """
)

instructions = widgets.HTML(
    value="""
    <div style="
        background: #f3f4f6;
        color: #111827;
        padding: 12px;
        border-radius: 10px;
        font-family: Arial;
        margin-bottom: 12px;
        border-left: 5px solid #2563eb;
        width: 650px;
    ">
        <b>Workflow:</b><br>
        1. Click a tool button.<br>
        2. Select file(s) from your PC.<br>
        3. The selected files will appear below.<br>
        4. Use <b>↑</b> and <b>↓</b> to change the order if needed.<br>
        5. Click <b>Process</b> to run the selected action.<br>
        6. Click <b>Download File</b> and choose where to save the result.
    </div>
    """
)

btn_merge = widgets.Button(
    description="Merge PDFs",
    icon="files-o",
    button_style="primary",
    layout=widgets.Layout(width="210px", height="45px")
)

btn_split = widgets.Button(
    description="Split PDF",
    icon="cut",
    button_style="info",
    layout=widgets.Layout(width="210px", height="45px")
)

btn_compress = widgets.Button(
    description="Compress PDFs",
    icon="compress",
    button_style="warning",
    layout=widgets.Layout(width="210px", height="45px")
)

btn_jpg_to_pdf = widgets.Button(
    description="JPG to PDF",
    icon="image",
    button_style="success",
    layout=widgets.Layout(width="210px", height="45px")
)

btn_pdf_to_jpg = widgets.Button(
    description="PDF to JPG",
    icon="file-image-o",
    button_style="danger",
    layout=widgets.Layout(width="210px", height="45px")
)

btn_merge.on_click(lambda b: start_action("Merge PDFs"))
btn_split.on_click(lambda b: start_action("Split PDF"))
btn_compress.on_click(lambda b: start_action("Compress PDFs"))
btn_jpg_to_pdf.on_click(lambda b: start_action("JPG to PDF"))
btn_pdf_to_jpg.on_click(lambda b: start_action("PDF to JPG"))

tool_buttons_row1 = widgets.HBox(
    [btn_merge, btn_split, btn_compress],
    layout=widgets.Layout(gap="12px", margin="8px 0")
)

tool_buttons_row2 = widgets.HBox(
    [btn_jpg_to_pdf, btn_pdf_to_jpg],
    layout=widgets.Layout(gap="12px", margin="8px 0")
)

message_html = widgets.HTML()

status_html = widgets.HTML()

selected_files_title = widgets.HTML(
    value="""
    <h3 style="font-family: Arial; margin-top: 16px; margin-bottom: 8px;">
        📁 Selected Files
    </h3>
    """
)

file_list_box = widgets.VBox()
action_buttons_box = widgets.VBox()
download_box = widgets.VBox()

app = widgets.VBox(
    [
        title,
        instructions,
        tool_buttons_row1,
        tool_buttons_row2,
        message_html,
        status_html,
        selected_files_title,
        file_list_box,
        action_buttons_box,
        download_box
    ],
    layout=widgets.Layout(width="100%")
)

display(app)

update_status_text()
update_file_list_display()
set_message("PDF Tools GUI is ready. Choose a tool to start.", "success")